# MediVision AI - Kaggle training notebook (TensorFlow / Keras)

**Self-contained:** this notebook does *not* need `train.py` or any project folder. Everything is defined in the cells below.
Educational / research use only. Nothing here is a medical diagnosis.

**Before you run**
1. Right sidebar -> **Input** -> **Add Input**, then attach the datasets you want to train (search by title):
   * *Chest X-Ray Images (Pneumonia)* (Paul Mooney) -> pneumonia module
   * *Brain Tumor MRI Dataset* (Masoud Nickparvar) -> brain tumor module
   * Competition *APTOS 2019 Blindness Detection* (join and accept the rules first) -> diabetic retinopathy module
   * *Breast Ultrasound Images Dataset* (BUSI, by Arya Shah) -> breast cancer module (downloaded automatically if you do not attach it; needs Internet ON)
2. Settings -> Accelerator: **GPU** (recommended) and Internet **On** (needed to download ImageNet weights).
3. Run the cells **top to bottom**. Cells 1-4 only inspect the environment and datasets and never train anything.
4. The last cells (22-25) start the full-stack website (FastAPI + React) on a public link, so you can upload images, choose a module or use Auto-detect, and see prediction + Grad-CAM. Training must have succeeded first.

**What was wrong with the previous notebook (simple explanation)**
* `next(...)` was called on a search for `training/train.py`. Only the notebook itself was attached, so the search found nothing and Python raised `StopIteration` (`next()` on an empty result, with no default value).
* The notebook relied on files that were never uploaded (training scripts, backend), so it could not work on its own.
* It silently skipped missing datasets and could still create an (empty) zip, which looks like success when nothing was trained.
* It picked the first matching folder without checking it, never checked the environment, and crashed if ImageNet weights could not be downloaded.

All of these are fixed below: no `next()` without a fallback, no external files, explicit NOT CONFIGURED reporting, validated dataset discovery, and a zip only after verified exports.

In [ ]:
# ===== CELL 1: environment verification =====
import os, sys, gc, platform, shutil, glob, json, random, time, re, zipfile, hashlib, traceback, datetime, warnings
from pathlib import Path
import subprocess

def _need(module, pip_name=None):
    try: __import__(module)
    except ImportError:
        print(f"installing missing dependency: {pip_name or module}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or module])
for m, p in [("numpy", "numpy"), ("pandas", "pandas"), ("sklearn", "scikit-learn"), ("PIL", "pillow"), ("matplotlib", "matplotlib")]:
    _need(m, p)

import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers as L, applications as A

# Kaggle paths (override with env vars only for local testing)
INPUT_ROOT = Path(os.environ.get("MEDIVISION_INPUT", "/kaggle/input"))      # read-only
WORK = Path(os.environ.get("MEDIVISION_WORK", "/kaggle/working")) / "medivision"
for sub in ("models", "reports", "checkpoints", "webapp_weights"):
    (WORK / sub).mkdir(parents=True, exist_ok=True)

print("Python      :", sys.version.split()[0], "|", platform.platform())
print("TensorFlow  :", tf.__version__, "| Keras", keras.__version__)
GPUS = tf.config.list_physical_devices("GPU")
for g in GPUS:
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception as e: print("  (could not enable memory growth:", e, ")")
print("GPU devices :", [g.name for g in GPUS] if GPUS else "none -> CPU fallback (smaller epochs/batch, subsampled data; see config)")
try:
    free_gb = shutil.disk_usage(WORK).free / 1e9
    print(f"Free disk   : {free_gb:.1f} GB at {WORK}")
except Exception as e: print("disk check failed:", e)
try:
    (WORK / ".write_test").write_text("ok"); (WORK / ".write_test").unlink(); print("Output dir  : writable ->", WORK)
except Exception as e: print("OUTPUT DIR NOT WRITABLE:", e)
print("INPUT_ROOT  :", INPUT_ROOT, "| exists:", INPUT_ROOT.exists())
if INPUT_ROOT.exists(): print("Attached top-level inputs:", sorted(os.listdir(INPUT_ROOT)) or "NONE")
else: print("WARNING: input folder missing - are you running on Kaggle?")

def show_plot():
    if "agg" not in matplotlib.get_backend().lower(): plt.show()
    plt.close("all")

In [ ]:
# ===== CELL 2: configuration =====
SEED = 42
try: keras.utils.set_random_seed(SEED)
except Exception: random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ---- Dataset paths: leave None to auto-discover, or paste a verified path to override ----
PNEUMONIA_DIR   = None   # folder that contains train/ (val/) test/, each with NORMAL/ and PNEUMONIA/
BRAIN_TUMOR_DIR = None   # folder that contains Training/ and Testing/ with the class folders
DR_CSV_PATH     = None   # e.g. /kaggle/input/aptos2019-blindness-detection/train.csv
DR_IMAGE_DIR    = None   # e.g. /kaggle/input/aptos2019-blindness-detection/train_images
BREAST_DIR      = None   # folder with benign/, malignant/ and normal/ (the Dataset_BUSI_with_GT folder)

MODULES_TO_RUN  = ["pneumonia", "brain_tumor", "diabetic_retinopathy", "breast_cancer"]
SMOKE_TEST_ONLY = False          # True = verify data + one forward/backward pass, no training
USE_PRETRAINED  = True           # ImageNet weights; falls back to random init (with a loud warning) if unavailable
EPOCHS_HEAD, EPOCHS_FINE = (5, 15) if GPUS else (2, 2)    # stage 1 / stage 2 maximum epochs (early stopping may stop sooner)
BATCH_SIZE      = 32 if GPUS else 16
UNFREEZE_FRACTION = 0.30         # stage 2: fraction of backbone layers (from the top) that are fine-tuned
CPU_SUBSAMPLE_PER_CLASS = None if GPUS else 300   # CPU fallback: cap images per class (results then NOT comparable to full training)
IMG_SIZE_OVERRIDE = None         # only for quick debugging, e.g. 128

MODULES = {
    "pneumonia": dict(title="Pneumonia", input="Chest X-ray", arch="densenet121", size=224, flip=None,
                      patient_regex=[r"(person\d+)", r"(IM-\d{4})"], positive_regex=r"pneumonia", ordinal=False),
    "brain_tumor": dict(title="Brain Tumor", input="Brain MRI", arch="efficientnet_b0", size=224, flip=None,
                        positive_regex=None, ordinal=False),
    "diabetic_retinopathy": dict(title="Diabetic Retinopathy", input="Retinal fundus", arch="resnet50v2", size=256,
                                 flip="horizontal_and_vertical", positive_regex=None, ordinal=True),
    "breast_cancer": dict(title="Breast Cancer", input="Breast ultrasound", arch="efficientnet_b0", size=224, flip="horizontal",
                          positive_regex=r"malignant", ordinal=False),
}
DATASET_INFO = {
    "pneumonia": dict(title="Chest X-Ray Images (Pneumonia)", var="PNEUMONIA_DIR",
        expected="a folder with train/, test/ (and val/), each containing NORMAL/ and PNEUMONIA/ image folders",
        steps='Add Input -> search "Chest X-Ray Images (Pneumonia)" (Paul Mooney) -> Add.'),
    "brain_tumor": dict(title="Brain Tumor MRI Dataset", var="BRAIN_TUMOR_DIR",
        expected="a folder with Training/ and Testing/, each containing one folder per tumor class",
        steps='Add Input -> search "Brain Tumor MRI Dataset" (Masoud Nickparvar) -> Add.'),
    "diabetic_retinopathy": dict(title="APTOS 2019 Blindness Detection", var="DR_CSV_PATH / DR_IMAGE_DIR",
        expected="train.csv (columns id_code, diagnosis) and a folder of <id_code>.png images",
        steps='Join the competition "APTOS 2019 Blindness Detection" (accept the rules) -> Add Input -> Competition Data.'),
    "breast_cancer": dict(title="Breast Ultrasound Images Dataset (BUSI; Kaggle: aryashah2k/breast-ultrasound-images-dataset)", var="BREAST_DIR",
        expected="benign/, malignant/ and normal/ image folders (each image has a *_mask.png companion that is ignored)",
        steps='Normally downloaded automatically (Internet ON). Manual alternative: Add Input -> search "Breast Ultrasound Images Dataset" (Arya Shah) -> Add.'),
}
print("Config OK | epochs (head, fine):", (EPOCHS_HEAD, EPOCHS_FINE), "| batch:", BATCH_SIZE, "| CPU subsample cap:", CPU_SUBSAMPLE_PER_CLASS)

In [ ]:
# ===== CELL 3: dataset discovery helpers (no training here) =====
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
SPLIT_ALIASES = {"train": "train", "training": "train", "val": "val", "valid": "val", "validation": "val", "test": "test", "testing": "test"}
def is_img(p): return Path(p).suffix.lower() in IMG_EXT
def _skip(name): return name.startswith(("__", "."))

def walk(root, max_depth=6):
    # yields (dir, subdirs, files) for a directory tree; safe if root is missing
    root = Path(root)
    if not root.exists(): return
    stack = [(root, 0)]
    while stack:
        d, depth = stack.pop()
        try:
            with os.scandir(d) as it: entries = list(it)
        except OSError: continue
        subdirs = [Path(e.path) for e in entries if e.is_dir() and not _skip(e.name)]
        files = [Path(e.path) for e in entries if e.is_file()]
        yield d, subdirs, files
        if depth < max_depth:
            for s in sorted(subdirs, reverse=True): stack.append((s, depth + 1))

def count_images(d): return sum(1 for f in Path(d).rglob("*") if f.is_file() and is_img(f))
def has_image(d): return any(f.is_file() and is_img(f) for f in Path(d).rglob("*"))

def class_dirs(d):
    out = []
    for c in sorted(Path(d).iterdir()):
        if c.is_dir() and not _skip(c.name) and has_image(c): out.append(c)
    return out

def folder_layout(root):
    # ('split', {split: [class dirs]}) | ('flat', [class dirs]) | None
    root = Path(root)
    subs = [d for d in root.iterdir() if d.is_dir() and not _skip(d.name)]
    splits = {}
    for d in subs:
        if d.name.lower() in SPLIT_ALIASES:
            cd = class_dirs(d)
            if len(cd) >= 2: splits[SPLIT_ALIASES[d.name.lower()]] = cd
    if "train" in splits: return "split", splits
    flat = class_dirs(root)
    return ("flat", flat) if len(flat) >= 2 else None

def layout_classes(lay):
    kind, data = lay
    return [c.name for c in (data["train"] if kind == "split" else data)]

def pred_pneumonia(lay): return {"normal", "pneumonia"} <= {n.lower() for n in layout_classes(lay)}
def pred_brain(lay):
    kind, data = lay
    names = [n.lower() for n in layout_classes(lay)]
    return kind == "split" and "test" in data and any(k in n for n in names for k in ("glioma", "meningioma", "pituitary"))
def pred_breast(lay): return {"benign", "malignant", "normal"} <= {n.lower() for n in layout_classes(lay)}

def layout_signature(lay):
    kind, data = lay
    if kind == "split": return tuple(sorted((s, c.name.lower(), count_images(c)) for s, cs in data.items() for c in cs))
    return tuple(sorted((c.name.lower(), count_images(c)) for c in data))

def discover_folders(predicate, root=None):
    cands = []
    for d, subs, files in walk(root or INPUT_ROOT):
        if len(subs) < 2: continue
        try: lay = folder_layout(d)
        except OSError: continue
        if lay and predicate(lay): cands.append(d)
    outer = [c for c in cands if not any(o != c and o in c.parents for o in cands)]   # drop nested copies (e.g. chest_xray/chest_xray)
    groups = {}
    for c in outer: groups.setdefault(layout_signature(folder_layout(c)), []).append(c)
    return outer, groups

def resolve_folder_dataset(name, manual, predicate, root=None):
    if manual:
        p = Path(manual)
        lay = folder_layout(p) if p.is_dir() else None
        if lay and predicate(lay): return dict(status="OK", path=p, note="user-specified path", candidates=[p])
        return dict(status="NOT CONFIGURED", path=None, candidates=[], note=f"the path you set ({p}) does not look like the {name} dataset")
    outer, groups = discover_folders(predicate, root)
    if len(groups) == 1:
        paths = sorted(next(iter(groups.values())), key=lambda p: (len(p.parts), str(p)))
        extra = f" ({len(paths)} identical copies found; using the shortest path)" if len(paths) > 1 else ""
        return dict(status="OK", path=paths[0], note="auto-discovered" + extra, candidates=outer)
    if not groups: return dict(status="NOT CONFIGURED", path=None, candidates=[], note="no matching dataset found in /kaggle/input")
    return dict(status="NOT CONFIGURED", path=None, candidates=outer,
                note=f"ambiguous: {len(groups)} different folders match; set {DATASET_INFO[name]['var']} to the right one")

def _ids_found(d, ids):
    return sum(any((Path(d) / f"{i}{e}").exists() for e in (".png", ".jpg", ".jpeg")) for i in ids) / max(1, len(ids))

def resolve_dr(csv_manual, img_manual):
    csvs = [Path(csv_manual)] if csv_manual else []
    if not csvs:
        for d, subs, files in walk(INPUT_ROOT):
            for f in files:
                if f.suffix.lower() == ".csv":
                    try: cols = set(pd.read_csv(f, nrows=1).columns)
                    except Exception: continue
                    if {"id_code", "diagnosis"} <= cols: csvs.append(f)
    combos = []
    for c in csvs:
        if not c.exists(): continue
        ids = pd.read_csv(c, nrows=40)["id_code"].astype(str).tolist()
        dirs = [Path(img_manual)] if img_manual else [d for d, _, _ in walk(c.parent, max_depth=3)]
        for d in dirs:
            if d.is_dir() and _ids_found(d, ids) >= 0.9: combos.append((c, d)); break
    if len(combos) == 1: return dict(status="OK", csv=combos[0][0], img_dir=combos[0][1], note="user-specified" if csv_manual else "auto-discovered", candidates=csvs)
    if not combos: return dict(status="NOT CONFIGURED", csv=None, img_dir=None, candidates=csvs, note="no train.csv with matching images found")
    return dict(status="NOT CONFIGURED", csv=None, img_dir=None, candidates=csvs, note="ambiguous: several train.csv candidates; set DR_CSV_PATH and DR_IMAGE_DIR")

def show_tree(root, depth=3, indent=0, max_items=12):
    root = Path(root)
    try: entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name))
    except Exception as e: print(" " * indent + f"[cannot list {root}: {e}]"); return
    dirs = [e for e in entries if e.is_dir() and not _skip(e.name)]
    files = [e for e in entries if e.is_file()]
    imgs = [f for f in files if is_img(f)]
    for d in dirs[:max_items]:
        print(" " * indent + d.name + "/")
        if depth > 1: show_tree(d, depth - 1, indent + 2, max_items)
    if len(dirs) > max_items: print(" " * indent + f"... {len(dirs) - max_items} more folders")
    if imgs: print(" " * indent + f"({len(imgs)} image files)")
    for f in [f for f in files if not is_img(f)][:max_items]: print(" " * indent + f.name)
print("discovery helpers defined")

In [ ]:
# ===== CELL 4: run discovery, show what was found, report missing datasets =====
print("Directory tree of", INPUT_ROOT, "(depth 3):")
show_tree(INPUT_ROOT, depth=3)

DATASETS = {
    "pneumonia": resolve_folder_dataset("pneumonia", PNEUMONIA_DIR, pred_pneumonia),
    "brain_tumor": resolve_folder_dataset("brain_tumor", BRAIN_TUMOR_DIR, pred_brain),
    "diabetic_retinopathy": resolve_dr(DR_CSV_PATH, DR_IMAGE_DIR),
    "breast_cancer": resolve_folder_dataset("breast_cancer", BREAST_DIR, pred_breast),
}
# Breast dataset: if none is attached, download the public Kaggle dataset automatically (needs Internet ON; check its licence on Kaggle)
AUTO_DOWNLOAD_BREAST = True
BREAST_KAGGLE_SLUG = "aryashah2k/breast-ultrasound-images-dataset"
if DATASETS["breast_cancer"]["status"] != "OK" and AUTO_DOWNLOAD_BREAST and "breast_cancer" in MODULES_TO_RUN and not BREAST_DIR:
    try:
        _need("kagglehub"); import kagglehub
        dl = Path(kagglehub.dataset_download(BREAST_KAGGLE_SLUG)); print("Downloaded breast dataset to", dl)
        DATASETS["breast_cancer"] = resolve_folder_dataset("breast_cancer", None, pred_breast, root=dl)
        DATASETS["breast_cancer"]["note"] += f" (downloaded from Kaggle: {BREAST_KAGGLE_SLUG})"
    except Exception as e:
        print(f"Automatic breast dataset download failed ({type(e).__name__}: {str(e)[:150]}). Turn Internet ON, or attach the dataset manually (see below).")
print("\n=== Dataset status ===")
for name, r in DATASETS.items():
    if r["status"] == "OK":
        where = r["path"] if name != "diabetic_retinopathy" else f"{r['csv']}  +  {r['img_dir']}"
        print(f"[ OK ] {name:22s} {where}  ({r['note']})")
    else:
        info = DATASET_INFO[name]
        print(f"[ NOT CONFIGURED ] {name}")
        print("    Expected dataset :", info["title"])
        print("    Expected layout  :", info["expected"])
        print("    Problem          :", r["note"])
        print("    Paths discovered :", [str(c) for c in r["candidates"]] or "none (attached inputs: %s)" % (sorted(os.listdir(INPUT_ROOT)) if INPUT_ROOT.exists() else "none"))
        print("    How to attach    :", info["steps"], f"Then re-run this cell (or set {info['var']} in the config cell).")

In [ ]:
# ===== CELL 5: dataset loading, validation, splitting =====
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, confusion_matrix,
                             roc_auc_score, cohen_kappa_score)
from PIL import Image

DR_GRADES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]

class NotConfigured(Exception): pass

def pretty(n):
    if n.lower() == "notumor": return "No Tumor"
    return re.sub(r"^\d+_", "", n).replace("_", " ").strip().title()

def patient_id(path, spec):
    for rx in spec.get("patient_regex", []):
        m = re.search(rx, Path(path).name)
        if m: return m.group(1)
    return str(path)

def collect_folder_samples(root):
    kind, data = folder_layout(root)
    rows = []
    items = list(data.items()) if kind == "split" else [(None, data)]
    for split, cdirs in items:
        for c in cdirs:
            for f in c.rglob("*"):
                if f.is_file() and is_img(f) and "_mask" not in f.stem.lower(): rows.append((str(f), c.name.lower(), pretty(c.name), split))   # BUSI ships a *_mask.png next to every image
    return pd.DataFrame(rows, columns=["path", "label_key", "label_name", "split"])

def collect_dr_samples(csv_path, img_dir):
    d = pd.read_csv(csv_path)
    miss = {"id_code", "diagnosis"} - set(d.columns)
    if miss: raise ValueError(f"CSV is missing columns: {miss}")
    d = d.dropna(subset=["id_code", "diagnosis"]).copy()
    d["diagnosis"] = pd.to_numeric(d["diagnosis"], errors="coerce")
    bad = d[~d["diagnosis"].isin(range(5))]
    if len(bad): raise ValueError(f"{len(bad)} rows have a grade outside 0-4 (e.g. {bad['diagnosis'].unique()[:5]})")
    rows, missing = [], 0
    for i, g in zip(d["id_code"].astype(str), d["diagnosis"].astype(int)):
        p = next((Path(img_dir) / f"{i}{e}" for e in (".png", ".jpg", ".jpeg") if (Path(img_dir) / f"{i}{e}").exists()), None)
        if p is None: missing += 1; continue
        rows.append((str(p), str(g), DR_GRADES[g], None))
    if missing: print(f"  note: {missing} CSV rows have no image file and were skipped")
    return pd.DataFrame(rows, columns=["path", "label_key", "label_name", "split"])

def verify_images(df):
    # drops unreadable / too small images; returns (clean_df, [(path, reason)])
    good, bad = [], []
    for k, p in enumerate(df["path"]):
        try:
            with Image.open(p) as im:
                w, h = im.size
                if min(w, h) < 32: raise ValueError(f"too small ({w}x{h})")
                im.verify()
            good.append(True)
        except Exception as e:
            good.append(False); bad.append((p, f"{type(e).__name__}: {e}"))
        if (k + 1) % 3000 == 0: print(f"  verified {k + 1}/{len(df)} images")
    return df[np.array(good, dtype=bool)].copy(), bad

def check_class_consistency(df):
    if df["split"].notna().any():
        ref = set(df[df.split == "train"].label_key)
        for s in df["split"].dropna().unique():
            ks = set(df[df.split == s].label_key)
            if ks != ref: raise ValueError(f"split '{s}' has classes {sorted(ks)} but train has {sorted(ref)} (unexpected class names)")

def carve(df, frac, seed=SEED):
    k = max(2, int(round(1 / frac)))
    rest, held = next(iter(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed).split(df, df["label"], groups=df["group"])))
    return df.iloc[rest], df.iloc[held]

def make_splits(df, spec):
    df = df.copy()
    df["group"] = [patient_id(p, spec) for p in df["path"]]
    share = float(np.mean([g != p for g, p in zip(df["group"], df["path"])]))
    notes = []
    if df["split"].notna().any():
        tr, va, te = (df[df.split == s] for s in ("train", "val", "test"))
        if len(te) == 0:
            pool = pd.concat([tr, va]); tr, te = carve(pool, 0.15); va = pool.iloc[0:0]
            notes.append("no official test split: 15% of the data carved out as test")
        else: notes.append("official test split preserved")
        if len(va) < 0.10 * len(tr):
            pool = pd.concat([tr, va]); tr, va = carve(pool, 0.15)
            notes.append("official validation split missing/too small: 15% of training data carved out as validation")
        else: notes.append("official validation split preserved")
    else:
        rest, te = carve(df, 0.15); tr, va = carve(rest, 0.15)
        notes.append("no official splits: stratified ~70/15/15")
    out = pd.concat([tr.assign(split="train"), va.assign(split="val"), te.assign(split="test")])
    notes.append(f"patient-level grouping from filenames for {share:.0%} of images" if share > 0.5 else "image-level split (no patient identifiers available) - metrics may be optimistic if one patient has several images")
    return out, dict(split_notes=notes, patient_group_share=share)

def quick_hash(p):
    with open(p, "rb") as f: b = f.read(1 << 20)
    return hashlib.md5(b + str(os.path.getsize(p)).encode()).hexdigest()

def remove_leakage(df):
    df = df.copy()
    if "hash" not in df.columns: df["hash"] = [quick_hash(p) for p in df["path"]]
    tr_h = set(df[df.split == "train"].hash); va_h = set(df[df.split == "val"].hash)
    leak = ((df.split.isin(["val", "test"])) & df.hash.isin(tr_h)) | ((df.split == "test") & df.hash.isin(va_h))
    return df[~leak], int(leak.sum())

def subsample(df, cap):
    if not cap: return df, False
    out = []
    for s, c in (("train", cap), ("val", max(30, cap // 2)), ("test", max(30, cap // 2))):
        out.append(df[df.split == s].sample(frac=1, random_state=SEED).groupby("label").head(c))
    return pd.concat(out), True

def split_problem(df, n_classes):
    for s_ in ("train", "val", "test"):
        got = set(df[df.split == s_].label)
        if got != set(range(n_classes)): return f"split '{s_}' is missing classes {sorted(set(range(n_classes)) - got)}"
    return None

def prepare_dataset(name):
    spec, res = MODULES[name], DATASETS[name]
    if res["status"] != "OK": raise NotConfigured(res["note"])
    df = collect_dr_samples(res["csv"], res["img_dir"]) if name == "diabetic_retinopathy" else collect_folder_samples(res["path"])
    rep = {"images_found": int(len(df))}
    if df.empty: raise ValueError("dataset is empty (no readable images found)")
    df, bad = verify_images(df); rep["corrupt_removed"] = len(bad); rep["corrupt_examples"] = bad[:5]
    if df.empty: raise ValueError("all images were unreadable")
    check_class_consistency(df)
    keys = sorted(df["label_key"].unique())
    if len(keys) < 2: raise ValueError(f"need at least 2 classes, found {keys}")
    names = {k: df[df.label_key == k].label_name.iloc[0] for k in keys}
    classes = [names[k] for k in keys]; n = len(classes)
    df["label"] = df["label_key"].map({k: i for i, k in enumerate(keys)})
    tiny = {names[k]: int((df.label_key == k).sum()) for k in keys if (df.label_key == k).sum() < 20}
    if tiny: raise ValueError(f"classes with fewer than 20 images: {tiny}")
    df["hash"] = [quick_hash(p) for p in df["path"]]
    conflict = df.groupby("hash")["label_key"].nunique(); bad_h = set(conflict[conflict > 1].index)
    if bad_h: df = df[~df.hash.isin(bad_h)].copy(); rep["conflicting_duplicates_removed"] = int(len(bad_h))      # same image filed under two classes
    rep["exact_duplicate_images"] = int(len(df) - df["hash"].nunique())
    base = df.copy()
    df, r2 = make_splits(base, spec); rep.update(r2)
    df, nleak = remove_leakage(df); rep["duplicates_removed_from_val_test"] = nleak
    problem = split_problem(df, n)
    if problem:
        # The official/random split is unusable (typical cause: many val/test images are exact copies of training images).
        rep["first_attempt_problem"] = f"{problem} (after removing {nleak} val/test images that were exact copies of training images)"
        pool = base.drop_duplicates("hash").copy(); pool["group"] = [patient_id(p, spec) for p in pool["path"]]
        rest, te = carve(pool, 0.15); tr, va = carve(rest, 0.15)
        df = pd.concat([tr.assign(split="train"), va.assign(split="val"), te.assign(split="test")])
        rep["split_notes"] = [f"official splits unusable ({problem}); all UNIQUE images were pooled and re-split ~70/15/15 (stratified)",
                              "near-duplicate copies (flipped / rotated / re-exported) cannot be detected exactly and may still cross splits - treat the metrics as optimistic"]
        rep["resplit_pooled"] = True; rep["duplicates_removed_from_val_test"] = int(len(base) - len(pool))
        problem = split_problem(df, n)
        if problem: raise ValueError(f"{problem} even after re-splitting the unique images: some class has too few unique images")
    df, sub = subsample(df, CPU_SUBSAMPLE_PER_CLASS); rep["subsampled"] = sub
    problem = split_problem(df, n)
    if problem: raise ValueError(problem + " after CPU subsampling - raise CPU_SUBSAMPLE_PER_CLASS")
    df = df.drop(columns=["hash"])
    rep["counts"] = {s_: int((df.split == s_).sum()) for s_ in ("train", "val", "test")}
    rep["per_class_train"] = {c: int(((df.split == "train") & (df.label == i)).sum()) for i, c in enumerate(classes)}
    rep["per_split_class"] = pd.crosstab(df["split"], df["label_name"]).to_dict("index")
    return dict(df=df.reset_index(drop=True), classes=classes, report=rep)

def size_for(name): return IMG_SIZE_OVERRIDE or MODULES[name]["size"]

def make_ds(df, size, batch, training):
    ds = tf.data.Dataset.from_tensor_slices((df["path"].values.astype(str), df["label"].values.astype("int32")))
    if training: ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    def load(p, y):
        img = tf.io.decode_image(tf.io.read_file(p), channels=3, expand_animations=False)
        img = tf.image.resize(img, (size, size)); img.set_shape((size, size, 3))
        return img, y
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.ignore_errors() if hasattr(ds, "ignore_errors") else ds.apply(tf.data.experimental.ignore_errors())
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)
print("data functions defined")

In [ ]:
# ===== CELL 6: model definition and two-stage transfer learning =====
BASE = {"densenet121": A.DenseNet121, "efficientnet_b0": A.EfficientNetB0, "resnet50v2": A.ResNet50V2}
PRE = {   # models take raw 0-255 RGB; the scaling each backbone needs lives INSIDE the model
    "densenet121": lambda: [L.Rescaling(1 / 255), L.Normalization(mean=[.485, .456, .406], variance=[.229**2, .224**2, .225**2])],
    "resnet50v2": lambda: [L.Rescaling(1 / 127.5, offset=-1)],
    "efficientnet_b0": lambda: []}    # EfficientNet has built-in rescaling

def build_model(arch, n_classes, size, flip=None, pretrained=True):
    bb, used = None, False
    if pretrained:
        try:
            bb = BASE[arch](include_top=False, weights="imagenet", input_shape=(size, size, 3)); used = True
        except Exception as e:
            print(f"  WARNING: ImageNet weights unavailable ({type(e).__name__}: {str(e)[:100]}). Using RANDOM initialisation - expect much lower accuracy. Turn Internet ON in Kaggle settings.")
    if bb is None: bb = BASE[arch](include_top=False, weights=None, input_shape=(size, size, 3))
    try: bb.name = "backbone"
    except AttributeError: bb._name = "backbone"
    inp = keras.Input((size, size, 3), name="image"); x = inp
    aug = [L.RandomRotation(0.05), L.RandomZoom(0.1), L.RandomContrast(0.1)]
    if flip: aug.insert(0, L.RandomFlip(flip))
    for l in aug + PRE[arch](): x = l(x)      # augmentation layers are inactive at inference
    x = bb(x, training=False)
    x = L.GlobalAveragePooling2D(name="gap")(x)
    x = L.Dropout(0.3, name="head_dropout")(x)
    out = L.Dense(n_classes, activation="softmax", name="predictions")(x)
    return keras.Model(inp, out, name=f"medivision_{arch}"), bb, used

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

def unfreeze_top(bb, frac):
    bb.trainable = True
    layers = bb.layers; n = max(1, int(len(layers) * frac))
    for l in layers[:-n]: l.trainable = False
    for l in layers:
        if isinstance(l, L.BatchNormalization): l.trainable = False     # keep ImageNet batch-norm statistics
    return sum(1 for l in layers if l.trainable)

def fit_stage(model, train_ds, val_ds, cw, epochs, ckpt, es_patience, lr_patience):
    cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=es_patience, restore_best_weights=True, verbose=1),
           keras.callbacks.ModelCheckpoint(str(ckpt), monitor="val_loss", save_best_only=True, save_weights_only=True),
           keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=lr_patience, min_lr=1e-7, verbose=1)]
    h = model.fit(train_ds, validation_data=val_ds, epochs=epochs, class_weight=cw, callbacks=cbs, verbose=2)
    return {k: [float(v) for v in vs] for k, vs in h.history.items()}
print("model functions defined")

In [ ]:
# ===== CELL 7: evaluation, plots, experiment tracking =====
def jsonable(o):
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, Path): return str(o)
    return str(o)
def save_json(obj, path): Path(path).write_text(json.dumps(obj, indent=2, default=jsonable))

def predict_ds(model, ds):
    ys, ps = [], []
    for x, y in ds:
        ps.append(model(x, training=False).numpy()); ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(ps)

def find_positive(classes, spec):
    rx = spec.get("positive_regex")
    if not rx: return None
    for i, c in enumerate(classes):
        if re.search(rx, c, re.I): return i
    return None

def compute_metrics(y, p, classes, spec, split_name):
    n = len(classes); labels = list(range(n)); pred = p.argmax(1)
    out = {"split": split_name, "n": int(len(y)), "accuracy": float(accuracy_score(y, pred))}
    pr, rc, f1, _ = precision_recall_fscore_support(y, pred, labels=labels, average="macro", zero_division=0)
    out.update(precision_macro=float(pr), recall_macro=float(rc), f1_macro=float(f1))
    P, R, F, S = precision_recall_fscore_support(y, pred, labels=labels, average=None, zero_division=0)
    cm = confusion_matrix(y, pred, labels=labels); per = {}
    for i, c in enumerate(classes):
        tp = int(cm[i, i]); fn = int(cm[i].sum() - tp); fp = int(cm[:, i].sum() - tp); tn = int(cm.sum() - tp - fn - fp)
        per[c] = dict(precision=float(P[i]), recall=float(R[i]), f1=float(F[i]), support=int(S[i]),
                      sensitivity=(tp / (tp + fn)) if tp + fn else None, specificity=(tn / (tn + fp)) if tn + fp else None)
    out["per_class"] = per; out["confusion_matrix"] = cm.tolist()
    pos = find_positive(classes, spec)
    try:
        if n == 2:
            k = pos if pos is not None else 1
            out["roc_auc"] = float(roc_auc_score((np.asarray(y) == k).astype(int), p[:, k]))
        else:
            out["roc_auc"] = float(roc_auc_score(y, p, multi_class="ovr", average="macro", labels=labels))
    except ValueError as e:
        out["roc_auc"] = None; out["roc_auc_note"] = str(e)
    if pos is not None:
        out["positive_class"] = classes[pos]; out["sensitivity"] = per[classes[pos]]["sensitivity"]; out["specificity"] = per[classes[pos]]["specificity"]
    if spec.get("ordinal"):
        out["quadratic_weighted_kappa"] = float(cohen_kappa_score(y, pred, weights="quadratic"))
    return out

def save_confusion_png(cm, classes, path, title):
    cm = np.array(cm); fig, ax = plt.subplots(figsize=(1.2 * len(classes) + 3, 1.0 * len(classes) + 2.5))
    ax.imshow(cm, cmap="Blues"); ax.set_xticks(range(len(classes))); ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=45, ha="right"); ax.set_yticklabels(classes)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    for i in range(len(classes)):
        for j in range(len(classes)): ax.text(j, i, int(cm[i, j]), ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    fig.tight_layout(); fig.savefig(path, dpi=130); show_plot()

def save_curves_png(history, path, title):
    h = {k: sum((history[s].get(k, []) for s in ("stage1", "stage2")), []) for k in ("loss", "val_loss", "accuracy", "val_accuracy")}
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5)); b = len(history["stage1"].get("loss", []))
    for a, (k1, k2) in zip(ax, (("loss", "val_loss"), ("accuracy", "val_accuracy"))):
        a.plot(h[k1], label=k1); a.plot(h[k2], label=k2); a.axvline(b - 0.5, color="gray", ls="--", lw=0.8); a.legend(); a.set_xlabel("epoch (dashed = start of fine-tuning)")
    fig.suptitle(title); fig.tight_layout(); fig.savefig(path, dpi=130); show_plot()

def log_experiment(row):
    f = WORK / "reports" / "experiments.csv"
    pd.concat([pd.read_csv(f), pd.DataFrame([row])]).to_csv(f, index=False) if f.exists() else pd.DataFrame([row]).to_csv(f, index=False)
print("evaluation functions defined")

In [ ]:
# ===== CELL 8: export, manifest, packaging =====
INFERENCE_PY = '''# Standalone inference helpers for MediVision AI artifacts (TensorFlow/Keras).
# Usage:
#     from inference import load_module, predict
#     bundle = load_module("models/pneumonia")
#     print(predict(bundle, "xray.jpg"))
# Educational/research use only. Not a medical diagnosis.
import json
from pathlib import Path
import numpy as np
import tensorflow as tf
from PIL import Image
from tensorflow import keras

def load_module(module_dir):
    d = Path(module_dir)
    model = keras.models.load_model(d / "model.keras", compile=False)
    classes = json.loads((d / "class_mapping.json").read_text())["classes"]
    size = json.loads((d / "preprocessing.json").read_text())["input_size"]
    cal = d / "calibration.json"
    temperature = json.loads(cal.read_text()).get("temperature", 1.0) if cal.exists() else 1.0
    return {"model": model, "classes": classes, "size": size, "temperature": temperature}

def load_image(path, size):
    arr = np.asarray(Image.open(path).convert("RGB"), dtype=np.float32)
    return tf.image.resize(arr, (size, size)).numpy()[None]          # same resize as training

def predict(bundle, path):
    x = load_image(path, bundle["size"])
    p = bundle["model"].predict(x, verbose=0)[0]
    z = np.log(np.clip(p, 1e-12, 1)) / bundle["temperature"]; z -= z.max(); q = np.exp(z) / np.exp(z).sum()
    i = int(q.argmax())
    return {"prediction": bundle["classes"][i], "confidence": float(q[i]),
            "probabilities": {c: float(v) for c, v in zip(bundle["classes"], q)},
            "note": "AI-assisted output for education/research; not a medical diagnosis."}

def _loader(name):
    return lambda base=".": load_module(Path(base) / "models" / name)
load_pneumonia = _loader("pneumonia"); load_brain_tumor = _loader("brain_tumor")
load_diabetic_retinopathy = _loader("diabetic_retinopathy"); load_breast_cancer = _loader("breast_cancer")
'''

ARTIFACT_README = '''MediVision AI artifacts (educational/research use only; not a medical diagnosis)

models/<module>/model.keras          full Keras model (accepts RAW RGB float32 0-255, shape (N, size, size, 3); scaling is inside the model)
models/<module>/class_mapping.json  exact class order of the softmax output
models/<module>/preprocessing.json   input size and resize rule (tf.image.resize bilinear)
models/<module>/model_config.json    architecture, training settings, split notes
models/<module>/metrics.json         validation AND test metrics (kept separate)
models/<module>/history.json         per-epoch training history
webapp_weights/<module>.json          metadata in the layout of the older MediVision back end (copy models/<module>/model.keras next to it as <module>.keras)
webapp/                              the full-stack website (FastAPI back end + built React front end); see webapp/README.md
reports/<module>/                    confusion matrix, curves, per-class metrics, splits.csv and any advanced analyses
manifest.json                        status of every module (SUCCESS / FAILED / NOT CONFIGURED)
inference.py                         load_module() and predict() helpers

Prediction output: {"prediction": str, "confidence": float, "probabilities": {class: float}, "note": str}
Example: from inference import load_pneumonia, predict; print(predict(load_pneumonia("."), "image.jpg"))
'''

def export_module(name, out, prep):
    spec, info = MODULES[name], DATASET_INFO[name]
    final = WORK / "models" / name; tmp = WORK / "models" / f".{name}.tmp"
    if tmp.exists(): shutil.rmtree(tmp)
    tmp.mkdir(parents=True)
    classes, size = prep["classes"], size_for(name)
    inf, _, _ = build_model(spec["arch"], len(classes), size, spec["flip"], pretrained=False)   # fresh uncompiled copy: no optimizer state (about 1/3 of the size)
    inf.set_weights(out["model"].get_weights()); inf.save(tmp / "model.keras")
    save_json({"classes": classes, "class_to_index": {c: i for i, c in enumerate(classes)}, "index_to_class": {str(i): c for i, c in enumerate(classes)}}, tmp / "class_mapping.json")
    save_json({"input_size": size, "channels": 3, "color_mode": "rgb", "input_range": "raw float32 0-255",
               "resize": "tf.image.resize bilinear (no antialias)",
               "scaling_inside_model": {"densenet121": "x/255 then ImageNet mean/std", "resnet50v2": "x/127.5 - 1", "efficientnet_b0": "built-in rescaling"}[spec["arch"]],
               "output": "softmax probabilities in class_mapping order"}, tmp / "preprocessing.json")
    save_json(out["config"], tmp / "model_config.json")
    save_json({"validation": out["val_metrics"], "test": out["test_metrics"]}, tmp / "metrics.json")
    save_json(out["history"], tmp / "history.json")
    m2 = keras.models.load_model(tmp / "model.keras", compile=False)          # integrity check: reload + compare
    a = out["model"](out["verify_x"], training=False).numpy(); b = m2(out["verify_x"], training=False).numpy()
    diff = float(np.abs(a - b).max())
    if diff > 1e-3: raise RuntimeError(f"exported model disagrees with the trained model (max diff {diff:.2e})")
    if final.exists(): shutil.rmtree(final)
    os.replace(tmp, final)
    tm = out["test_metrics"]                                                   # web-app compatible copy
    wa = WORK / "webapp_weights"   # metadata only; the model file lives in models/<module>/model.keras (the website reads it from there)
    save_json({"classes": classes, "dataset": info["title"], "version": "v1.0", "arch": spec["arch"], "size": size, "framework": "tensorflow",
               "metrics": {"accuracy": tm["accuracy"], "precision": tm["precision_macro"], "recall": tm["recall_macro"], "f1": tm["f1_macro"], "roc_auc": tm.get("roc_auc")}}, wa / f"{name}.json")
    return str(final), diff

def write_manifest(results):
    env = dict(tensorflow=tf.__version__, keras=keras.__version__, gpus=[g.name for g in GPUS], python=sys.version.split()[0])
    mods = {}
    for n, r in results.items():
        mods[n] = {k: r.get(k) for k in ("status", "reason", "dataset_path", "counts", "classes", "arch", "pretrained_weights_used", "subsampled",
                                         "model_dir", "epochs_run", "test_summary", "split_notes", "note")}
    adv = {n: sorted(p.name for p in (WORK / "models" / n).glob("*") if p.name in ("calibration.json", "robustness.json", "modality_gate.npz", "MODEL_CARD.md", "quality_profile.json")) for n in results if (WORK / "models" / n).exists()}
    manifest = dict(created=datetime.datetime.now().isoformat(timespec="seconds"), environment=env, modules=mods, advanced_artifacts=adv,
                    disclaimer="Educational/research use only. Not a medical diagnosis.")
    save_json(manifest, WORK / "manifest.json"); return manifest

def build_zip():
    (WORK / "inference.py").write_text(INFERENCE_PY); (WORK / "README_ARTIFACTS.txt").write_text(ARTIFACT_README)
    tmp = WORK.parent / "medivision_ai_artifacts.zip.tmp"; final = WORK.parent / "medivision_ai_artifacts.zip"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for f in [WORK / "manifest.json", WORK / "inference.py", WORK / "README_ARTIFACTS.txt"]: z.write(f, f.relative_to(WORK))
        for sub in ("models", "reports", "webapp_weights", "webapp"):
            for f in sorted((WORK / sub).rglob("*")):
                if f.is_file() and not f.name.startswith(".") and not f.name.endswith(".log"): z.write(f, f.relative_to(WORK))
    with zipfile.ZipFile(tmp) as z:
        bad = z.testzip(); names = z.namelist()
    if bad is not None: raise RuntimeError(f"corrupt entry in archive: {bad}")
    os.replace(tmp, final)
    return final, len(names), final.stat().st_size / 1e6
print("export functions defined")

In [ ]:
# ===== CELL 9: training controller =====
PREP, SMOKE, RESULTS = {}, {}, {}

def smoke_test(name, prep, bs=4):
    spec = MODULES[name]; size = size_for(name); n = len(prep["classes"])
    small = prep["df"][prep["df"].split == "train"].groupby("label").head(2)
    x, y = next(iter(make_ds(small, size, bs, False)))
    model, bb, used = build_model(spec["arch"], n, size, spec["flip"], USE_PRETRAINED)
    p = model(x, training=False).numpy()
    assert p.shape == (x.shape[0], n), f"output shape {p.shape}, expected (batch, {n})"
    assert np.isfinite(p).all() and abs(p.sum(1) - 1).max() < 1e-3, "model output is not a valid probability vector"
    compile_model(model, 1e-4); loss = float(model.train_on_batch(x, y)[0]); assert np.isfinite(loss), "training loss is not finite"
    del model; keras.backend.clear_session(); gc.collect()
    return dict(ok=True, input_shape=tuple(x.shape), pretrained=used, loss=loss)

def train_module(name, prep, bs):
    spec = MODULES[name]; size = size_for(name); classes = prep["classes"]; df = prep["df"]; t0 = time.time()
    tr, va, te = (df[df.split == s] for s in ("train", "val", "test"))
    train_ds, val_ds, test_ds = make_ds(tr, size, bs, True), make_ds(va, size, bs, False), make_ds(te, size, bs, False)
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.arange(len(classes)), y=tr["label"].values)))
    model, bb, used = build_model(spec["arch"], len(classes), size, spec["flip"], USE_PRETRAINED)
    ck = WORK / "checkpoints" / name; ck.mkdir(parents=True, exist_ok=True)
    bb.trainable = False; compile_model(model, 1e-3)
    print(f"-- stage 1: train the new head (backbone frozen), up to {EPOCHS_HEAD} epochs")
    h1 = fit_stage(model, train_ds, val_ds, cw, EPOCHS_HEAD, ck / "stage1.weights.h5", 3, 1)
    h2 = {}
    if EPOCHS_FINE > 0:
        n_tr = unfreeze_top(bb, UNFREEZE_FRACTION); compile_model(model, 1e-5)
        print(f"-- stage 2: fine-tune top {UNFREEZE_FRACTION:.0%} of backbone ({n_tr} trainable layers, batch-norm frozen), up to {EPOCHS_FINE} epochs")
        h2 = fit_stage(model, train_ds, val_ds, cw, EPOCHS_FINE, ck / "stage2.weights.h5", 4, 2)
    history = {"stage1": h1, "stage2": h2}
    yv, pv = predict_ds(model, val_ds); yt, pt = predict_ds(model, test_ds)
    vm = compute_metrics(yv, pv, classes, spec, "validation"); tm = compute_metrics(yt, pt, classes, spec, "test")
    rd = WORK / "reports" / name; rd.mkdir(parents=True, exist_ok=True)
    save_json({"validation": vm, "test": tm}, rd / "metrics.json")
    pd.DataFrame(tm["per_class"]).T.to_csv(rd / "per_class_test.csv")
    df.drop(columns=["label_key"]).to_csv(rd / "splits.csv", index=False)
    save_confusion_png(tm["confusion_matrix"], classes, rd / "confusion_matrix_test.png", f"{spec['title']} - TEST confusion matrix")
    save_curves_png(history, rd / "training_curves.png", f"{spec['title']} training")
    best_val = min(sum((h.get("val_loss", []) for h in (h1, h2)), []) or [float("nan")])
    cfg = dict(arch=spec["arch"], input_size=size, pretrained_weights_used=used, seed=SEED, batch_size=bs, loss="sparse_categorical_crossentropy (class-weighted)",
               optimizer="Adam", lr_stage1=1e-3, lr_stage2=1e-5, unfreeze_fraction=UNFREEZE_FRACTION, dropout=0.3, class_weights={classes[k]: float(v) for k, v in cw.items()},
               epochs_run={"stage1": len(h1.get("loss", [])), "stage2": len(h2.get("loss", []))}, best_val_loss=float(best_val),
               counts=prep["report"]["counts"], split_notes=prep["report"]["split_notes"], subsampled=prep["report"]["subsampled"],
               dataset=DATASET_INFO[name]["title"], train_seconds=round(time.time() - t0, 1), tensorflow=tf.__version__)
    vx = next(iter(test_ds))[0][:8]
    return dict(model=model, config=cfg, history=history, val_metrics=vm, test_metrics=tm, verify_x=vx)

def run_module(name):
    rec = RESULTS.setdefault(name, {"module": name})
    rec.update(status="FAILED", reason=None, arch=MODULES[name]["arch"])
    if name not in MODULES_TO_RUN: rec.update(status="NOT CONFIGURED", reason="not selected in MODULES_TO_RUN"); return rec
    res = DATASETS[name]
    if res["status"] != "OK": rec.update(status="NOT CONFIGURED", reason=res["note"]); return rec
    try:
        prep = PREP.get(name) or prepare_dataset(name); PREP[name] = prep
        rec.update(counts=prep["report"]["counts"], classes=prep["classes"], split_notes=prep["report"]["split_notes"], subsampled=prep["report"]["subsampled"],
                   dataset_path=str(res.get("path") or res.get("csv")))
        bs = BATCH_SIZE; out = None
        while out is None:
            try: out = train_module(name, prep, bs)
            except tf.errors.ResourceExhaustedError as e:
                keras.backend.clear_session(); gc.collect()
                if bs <= 2: raise
                bs //= 2; print(f"  GPU out of memory -> retrying with batch size {bs}")
        model_dir, diff = export_module(name, out, prep)
        tm = out["test_metrics"]
        rec.update(status="SUCCESS", reason=None, model_dir=model_dir, pretrained_weights_used=out["config"]["pretrained_weights_used"], epochs_run=out["config"]["epochs_run"],
                   test_summary={k: tm.get(k) for k in ("accuracy", "precision_macro", "recall_macro", "f1_macro", "roc_auc", "quadratic_weighted_kappa", "sensitivity", "specificity")},
                   note="trained on a subsample (CPU fallback): NOT comparable to full training" if rec["subsampled"] else None)
        log_experiment(dict(time=datetime.datetime.now().isoformat(timespec="seconds"), module=name, arch=MODULES[name]["arch"], batch=bs,
                            val_f1=out["val_metrics"]["f1_macro"], test_f1=tm["f1_macro"], test_acc=tm["accuracy"], seconds=out["config"]["train_seconds"]))
    except NotConfigured as e: rec.update(status="NOT CONFIGURED", reason=str(e))
    except KeyboardInterrupt:
        rec.update(status="FAILED", reason="interrupted by user"); raise
    except Exception as e:
        rec.update(status="FAILED", reason=f"{type(e).__name__}: {e}", traceback=traceback.format_exc())
        print(f"\n!! {name} FAILED: {type(e).__name__}: {e}\n{traceback.format_exc()}")
    finally:
        keras.backend.clear_session(); gc.collect()
    return rec
print("controller defined")

In [ ]:
# ===== CELL 10: PREPARE + SMOKE TEST every configured module (no training yet) =====
rows = []
for name in MODULES:
    rec = RESULTS.setdefault(name, {"module": name})
    if name not in MODULES_TO_RUN or DATASETS[name]["status"] != "OK":
        rows.append((name, "NOT CONFIGURED", DATASETS[name]["note"] if name in MODULES_TO_RUN else "not selected")); continue
    try:
        PREP[name] = prepare_dataset(name); r = PREP[name]["report"]
        print(f"\n[{name}] classes={PREP[name]['classes']}\n   images found={r['images_found']} corrupt removed={r['corrupt_removed']} duplicates removed={r['duplicates_removed_from_val_test']} subsampled={r['subsampled']}")
        print("   split sizes:", r["counts"], "| per-class train:", r["per_class_train"]); print("   split notes:", "; ".join(r["split_notes"]))
        print("   images per split/class:", r["per_split_class"])
        if r.get("first_attempt_problem"): print("   NOTE:", r["first_attempt_problem"], "-> re-split unique images")
        SMOKE[name] = smoke_test(name, PREP[name]); print("   smoke test OK:", SMOKE[name])
        rows.append((name, "SMOKE TEST PASSED (not trained)", ""))
    except Exception as e:
        rows.append((name, "FAILED", f"{type(e).__name__}: {e}")); print(f"\n[{name}] FAILED during preparation/smoke test:\n{traceback.format_exc()}")
print("\n=== Pre-flight summary ==="); print(pd.DataFrame(rows, columns=["module", "state", "detail"]).to_string(index=False))

In [ ]:
# ===== CELL 11: TRAIN + EVALUATE + EXPORT all configured modules =====
if SMOKE_TEST_ONLY: print("SMOKE_TEST_ONLY=True -> skipping training")
else:
    for name in MODULES:
        if name in SMOKE:
            print(f"\n{'=' * 14} {name} {'=' * 14}"); run_module(name)
        elif name in MODULES_TO_RUN and DATASETS[name]["status"] == "OK":
            RESULTS.setdefault(name, {"module": name}).update(status="FAILED", reason="pre-flight (cell 10) failed - see its output")
        else:
            run_module(name)       # records NOT CONFIGURED with the reason

In [ ]:
# ===== CELL 12: final summary, manifest, zip =====
for name in MODULES: RESULTS.setdefault(name, {"module": name, "status": "NOT CONFIGURED", "reason": "not run"})
if SMOKE_TEST_ONLY:
    for n, r in RESULTS.items():
        if n in SMOKE: r.update(status="NOT CONFIGURED", reason="smoke test only - nothing trained")
print("=== FINAL SUMMARY ===")
for n, r in RESULTS.items():
    print(f"{MODULES[n]['title']:22s}: {r['status']}" + (f"   ({r['reason']})" if r.get("reason") else ""))
ok = {n: r for n, r in RESULTS.items() if r["status"] == "SUCCESS"}
if ok:
    print("\nTEST-set metrics (held-out test split; macro-averaged):")
    print(pd.DataFrame({n: r["test_summary"] for n, r in ok.items()}).T.round(4).to_string())
    for n, r in ok.items():
        if r.get("subsampled"): print(f"  !! {n}: trained on a CPU subsample - do NOT report these numbers as final results")
        if not r.get("pretrained_weights_used"): print(f"  !! {n}: ImageNet weights were NOT used (random initialisation)")
    manifest = write_manifest(RESULTS); path, n_files, mb = build_zip()
    print(f"\nArchive written and verified: {path}  ({n_files} files, {mb:.1f} MB)")
    print("Download: Save Version -> Save & Run All (Commit) -> open the Output tab of the saved version -> medivision_ai_artifacts.zip")
    try:
        from IPython.display import FileLink, display; display(FileLink(str(path)))
    except Exception: pass
else:
    write_manifest(RESULTS); print("\nNo module finished successfully -> NO model files and NO archive were created. Fix the issues above and re-run cells 10-12.")

---
## Advanced improvements (run only after cell 12 reports at least one SUCCESS)
These cells re-load the **exported** models, so they also confirm that the exported files work. Each analysis is optional and saves its results next to the model.
Heatmaps show where the model looked; they do not prove disease. Model confidence is not a guaranteed probability of disease.

In [ ]:
# ===== CELL 13: helpers for advanced analyses =====
def trained_modules(): return [n for n, r in RESULTS.items() if r.get("status") == "SUCCESS"]
_CACHE = {}
def load_trained(name):
    if name not in _CACHE:
        d = WORK / "models" / name
        _CACHE[name] = dict(model=keras.models.load_model(d / "model.keras", compile=False), dir=d,
                            classes=json.loads((d / "class_mapping.json").read_text())["classes"],
                            size=json.loads((d / "preprocessing.json").read_text())["input_size"],
                            splits=pd.read_csv(WORK / "reports" / name / "splits.csv"))
    return _CACHE[name]

def mc_probs(model, x, passes=20):
    # Monte-Carlo dropout: only the head's dropout is stochastic; backbone features are computed once
    bb = model.get_layer("backbone"); i = model.layers.index(bb)
    pre = keras.Model(model.input, model.layers[i - 1].output)
    f = bb(pre(x, training=False), training=False); outs = []
    for _ in range(passes):
        y = f
        for l in model.layers[i + 1:]: y = l(y, training=True)
        outs.append(y.numpy())
    return np.stack(outs)

def split_probs(name, split, with_mc=False, passes=20, cap=None):
    b = load_trained(name); df = b["splits"][b["splits"].split == split]
    if cap: df = df.sample(frac=1, random_state=SEED).groupby("label").head(cap)
    ys, ps, mcs = [], [], []
    for x, y in make_ds(df, b["size"], BATCH_SIZE, False):
        ps.append(b["model"](x, training=False).numpy()); ys.append(y.numpy())
        if with_mc: mcs.append(mc_probs(b["model"], x, passes))
    res = (np.concatenate(ys), np.concatenate(ps))
    return res + ((np.concatenate(mcs, axis=1),) if with_mc else ())

def load_rgb(path): return np.asarray(Image.open(path).convert("RGB"), dtype=np.float32)
print("trained modules:", trained_modules() or "NONE - advanced cells will do nothing")

In [ ]:
# ===== CELL 14: Grad-CAM visual explanations =====
def gradcam(model, x):
    bb = model.get_layer("backbone"); i = model.layers.index(bb)
    pre = keras.Model(model.input, model.layers[i - 1].output); head = model.layers[i + 1:]
    with tf.GradientTape() as t:
        f = bb(pre(x, training=False), training=False); t.watch(f)
        y = f
        for l in head: y = l(y, training=False)
        idx = int(tf.argmax(y[0])); score = y[0, idx]
    g = t.gradient(score, f)
    cam = tf.nn.relu(tf.reduce_sum(f * tf.reduce_mean(g, (1, 2), keepdims=True), -1))[0].numpy()
    return y[0].numpy(), idx, (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

def overlay(rgb01, cam, alpha=0.4):
    h, w = rgb01.shape[:2]
    c = np.asarray(Image.fromarray((cam * 255).astype(np.uint8)).resize((w, h), Image.BILINEAR)) / 255.0
    return np.clip((1 - alpha) * rgb01 + alpha * plt.cm.jet(c)[..., :3], 0, 1)

for name in trained_modules():
    b = load_trained(name); te = b["splits"][b["splits"].split == "test"].sample(frac=1, random_state=SEED).head(6)
    fig, ax = plt.subplots(2, len(te), figsize=(2.6 * len(te), 5.8), squeeze=False)
    for k, (_, r) in enumerate(te.iterrows()):
        x = tf.image.resize(load_rgb(r["path"]), (b["size"], b["size"])).numpy()[None]
        p, idx, cam = gradcam(b["model"], x)
        ax[0, k].imshow(x[0].astype(np.uint8)); ax[0, k].set_title(f"true: {r['label_name']}", fontsize=8)
        ax[1, k].imshow(overlay(x[0] / 255.0, cam)); ax[1, k].set_title(f"pred: {b['classes'][idx]} ({p[idx]:.0%})", fontsize=8)
        for a in ax[:, k]: a.axis("off")
    fig.suptitle(f"{MODULES[name]['title']} - Grad-CAM = where the model looked (NOT proof of disease)", fontsize=9)
    fig.tight_layout(); fig.savefig(WORK / "reports" / name / "gradcam_examples.png", dpi=130); show_plot()
    print("saved", WORK / "reports" / name / "gradcam_examples.png")

In [ ]:
# ===== CELL 15: confidence calibration (validation data only) + uncertainty =====
def ece_score(p, y, bins=15):
    conf, correct = p.max(1), (p.argmax(1) == y); e = 0.0; edges = np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)
def apply_temperature(p, T):
    z = np.log(np.clip(p, 1e-12, 1)) / T; z -= z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
def nll(p, y): return float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)).mean())
def fit_temperature(p, y):
    grid = np.exp(np.linspace(np.log(0.1), np.log(10), 200)); return float(min(grid, key=lambda T: nll(apply_temperature(p, T), y)))
def entropy(p): return -(p * np.log(np.clip(p, 1e-12, 1))).sum(-1)

UNCERTAIN_CONF = 0.70        # heuristic: warn when calibrated top probability is below this
UNCERTAIN_ENTROPY = 0.60     # heuristic: warn when MC-dropout entropy exceeds this fraction of the maximum
def assess_prediction(q, mc_mean=None):
    n = len(q); warns = []
    if q.max() < UNCERTAIN_CONF: warns.append("Low confidence: the model is not sure about this image.")
    if mc_mean is not None and entropy(mc_mean) / np.log(n) > UNCERTAIN_ENTROPY: warns.append("High uncertainty: repeated stochastic passes disagree.")
    return {"uncertain": bool(warns), "warnings": warns}

CAL = {}
for name in trained_modules():
    yv, pv = split_probs(name, "val"); yt, pt, mc = split_probs(name, "test", with_mc=True)
    T = fit_temperature(pv, yv); qv, qt = apply_temperature(pv, T), apply_temperature(pt, T)
    mm = mc.mean(0); H = entropy(mm); err = mm.argmax(1) != yt
    flags = np.array([assess_prediction(qt[i], mm[i])["uncertain"] for i in range(len(yt))])
    acc = lambda m: float((qt.argmax(1)[m] == yt[m]).mean()) if m.any() else None
    res = dict(temperature=T, fitted_on="validation split only",
               ece_val_before=ece_score(pv, yv), ece_val_after=ece_score(qv, yv), ece_test_before=ece_score(pt, yt), ece_test_after=ece_score(qt, yt),
               nll_test_before=nll(pt, yt), nll_test_after=nll(qt, yt),
               mc_dropout_entropy_auroc_for_errors=(float(roc_auc_score(err, H)) if err.any() and (~err).any() else None),
               uncertain_fraction_test=float(flags.mean()), accuracy_when_flagged=acc(flags), accuracy_when_not_flagged=acc(~flags),
               thresholds=dict(low_confidence=UNCERTAIN_CONF, entropy_fraction=UNCERTAIN_ENTROPY, note="heuristic, not clinically validated"))
    CAL[name] = res; save_json(res, WORK / "models" / name / "calibration.json"); save_json(res, WORK / "reports" / name / "calibration.json")
    print(f"{name}: T={T:.2f} | test ECE {res['ece_test_before']:.3f} -> {res['ece_test_after']:.3f} | uncertain flagged on {flags.mean():.0%} of test images")

In [ ]:
# ===== CELL 16: image-quality validation + wrong-modality rejection (feature-space gate) =====
from sklearn.covariance import LedoitWolf

def quality_stats(img, size):
    g = np.asarray(img.convert("L").resize((size, size)), dtype=np.float32)
    lap = -4 * g[1:-1, 1:-1] + g[:-2, 1:-1] + g[2:, 1:-1] + g[1:-1, :-2] + g[1:-1, 2:]
    return dict(mean=float(g.mean()), std=float(g.std()), sharpness=float(lap.var()))

QUALITY = {}
for name in trained_modules():
    b = load_trained(name); tr = b["splits"][b["splits"].split == "train"].sample(frac=1, random_state=SEED).head(200)
    st = pd.DataFrame([quality_stats(Image.open(p).convert("RGB"), b["size"]) for p in tr["path"]])
    prof = dict(min_side=64, blank_std=2.0, too_dark_mean=float(st["mean"].quantile(0.005)), too_bright_mean=float(st["mean"].quantile(0.995)),
                low_contrast_std=float(st["std"].quantile(0.01)), blurry_sharpness=float(st["sharpness"].quantile(0.01)),
                note="thresholds are percentiles of this module's training images (data-driven heuristics)")
    QUALITY[name] = prof; save_json(prof, WORK / "models" / name / "quality_profile.json")

def quality_check(name, path):
    prof = QUALITY[name]; img = Image.open(path).convert("RGB"); w, h = img.size; s = quality_stats(img, load_trained(name)["size"]); issues, hard = [], False
    if min(w, h) < prof["min_side"]: issues.append(f"image is very small ({w}x{h})"); hard = True
    if s["std"] < prof["blank_std"]: issues.append("image looks blank"); hard = True
    if s["mean"] < prof["too_dark_mean"]: issues.append("image is unusually dark")
    if s["mean"] > prof["too_bright_mean"]: issues.append("image is unusually bright")
    if s["std"] < prof["low_contrast_std"]: issues.append("image has unusually low contrast")
    if s["sharpness"] < prof["blurry_sharpness"]: issues.append("image looks blurry")
    return {"ok": not hard, "issues": issues}

def gap_features(name, paths):
    b = load_trained(name); fm = keras.Model(b["model"].input, b["model"].get_layer("gap").output)
    df = pd.DataFrame({"path": list(paths), "label": 0})
    return np.concatenate([fm(x, training=False).numpy() for x, _ in make_ds(df, b["size"], BATCH_SIZE, False)])

def mahal(F, mu, P): d = F - mu; return ((d @ P) * d).sum(1)

GATE = {}
for name in trained_modules():
    b = load_trained(name); sp = b["splits"]
    Ftr = gap_features(name, sp[sp.split == "train"].sample(frac=1, random_state=SEED).head(1000)["path"])
    lw = LedoitWolf().fit(Ftr); mu, P = lw.location_, lw.precision_
    dval = mahal(gap_features(name, sp[sp.split == "val"].head(400)["path"]), mu, P); thr = float(np.quantile(dval, 0.99))
    GATE[name] = dict(mu=mu, P=P, thr=thr); np.savez(WORK / "models" / name / "modality_gate.npz", mean=mu, precision=P, threshold=thr)
print("gates fitted for:", list(GATE))

mods = list(GATE)
if len(mods) < 2: print("Wrong-modality evaluation needs at least 2 trained modules (gates were still saved).")
else:
    src = {m: load_trained(m)["splits"].query("split == 'test'").sample(frac=1, random_state=SEED).head(100)["path"].tolist() for m in mods}
    D = {(g, s): mahal(gap_features(g, src[s]), GATE[g]["mu"], GATE[g]["P"]) for g in mods for s in mods}
    rej = pd.DataFrame({s: {g: float((D[(g, s)] > GATE[g]["thr"]).mean()) for g in mods} for s in mods}); rej.index.name = "gate (rows) / image source (cols)"
    print("Fraction of images REJECTED by each gate (diagonal = false rejections of correct-modality images):"); print(rej.round(2).to_string())
    auc = {g: float(roc_auc_score(np.r_[np.zeros(len(D[(g, g)])), np.ones(sum(len(D[(g, s)]) for s in mods if s != g))], np.r_[D[(g, g)], *[D[(g, s)] for s in mods if s != g]])) for g in mods}
    route = {s: float(np.mean(np.argmin(np.stack([D[(g, s)] / GATE[g]["thr"] for g in mods]), 0) == mods.index(s))) for s in mods}
    print("AUROC (own modality vs other modalities):", {k: round(v, 3) for k, v in auc.items()}); print("Automatic module routing accuracy per source:", {k: round(v, 3) for k, v in route.items()})
    save_json(dict(rejection_matrix=rej.to_dict(), auroc=auc, routing_accuracy=route, threshold_rule="99th percentile of validation distances", note="evaluated on public-dataset test images only"), WORK / "reports" / "modality_gate.json")

In [ ]:
# ===== CELL 17: robustness testing (accuracy under image corruptions) =====
import io
def corrupt(arr, kind, lvl, rng):
    a = arr.astype(np.float32)
    if kind == "gaussian_noise": a = a + rng.normal(0, [8, 16, 32][lvl], a.shape)
    elif kind == "darker": a = a * [0.7, 0.5, 0.3][lvl]
    elif kind == "low_contrast": m = a.mean(); a = m + (a - m) * [0.7, 0.5, 0.3][lvl]
    elif kind == "blur":
        from PIL import ImageFilter
        return np.asarray(Image.fromarray(arr).filter(ImageFilter.GaussianBlur([1, 2, 4][lvl])))
    elif kind == "jpeg":
        buf = io.BytesIO(); Image.fromarray(arr).save(buf, "JPEG", quality=[50, 25, 10][lvl]); return np.asarray(Image.open(buf).convert("RGB"))
    return np.clip(a, 0, 255).astype(np.uint8)

KINDS = ["gaussian_noise", "blur", "darker", "low_contrast", "jpeg"]; ROB = {}
for name in trained_modules():
    b = load_trained(name); rng = np.random.default_rng(SEED)
    te = b["splits"][b["splits"].split == "test"].sample(frac=1, random_state=SEED).groupby("label").head(40)
    base = np.stack([tf.image.resize(load_rgb(p), (b["size"], b["size"])).numpy().astype(np.uint8) for p in te["path"]]); y = te["label"].values
    def score(X):
        pr = np.concatenate([b["model"](X[k:k + BATCH_SIZE].astype(np.float32), training=False).numpy().argmax(1) for k in range(0, len(X), BATCH_SIZE)])
        return float((pr == y).mean()), float(precision_recall_fscore_support(y, pr, labels=list(range(len(b["classes"]))), average="macro", zero_division=0)[2])
    a0, f0 = score(base); rows = [dict(corruption="clean", severity=0, accuracy=a0, f1_macro=f0, accuracy_drop=0.0)]
    for k in KINDS:
        for lvl in range(3):
            a, f = score(np.stack([corrupt(im, k, lvl, rng) for im in base])); rows.append(dict(corruption=k, severity=lvl + 1, accuracy=a, f1_macro=f, accuracy_drop=a0 - a))
    t = pd.DataFrame(rows); ROB[name] = t; t.to_csv(WORK / "reports" / name / "robustness.csv", index=False)
    save_json(dict(n_images=int(len(te)), note="corruptions applied at model resolution on a test subset", results=rows), WORK / "models" / name / "robustness.json")
    print(f"\n{name} (n={len(te)} test images) clean accuracy = {a0:.3f}; accuracy under corruption, columns = severity 1-3:"); print(t[t.corruption != "clean"].pivot(index="corruption", columns="severity", values="accuracy").round(3).to_string())

In [ ]:
# ===== CELL 18: cross-dataset evaluation (only if you provide an external dataset) =====
CROSS_DATASETS = {}   # e.g. {"pneumonia": "/kaggle/input/some-other-chest-xray-dataset"}  (class folders must be named like the training classes)
if not CROSS_DATASETS: print("Cross-dataset evaluation: NOT CONFIGURED (set CROSS_DATASETS above to run it).")
for name, path in CROSS_DATASETS.items():
    if name not in trained_modules(): print(f"{name}: not trained - skipped"); continue
    lay = folder_layout(path) if Path(path).is_dir() else None
    if not lay: print(f"{name}: {path} has no class folders - skipped"); continue
    b = load_trained(name); ext = collect_folder_samples(path); keymap = {c.lower().replace(" ", "_"): i for i, c in enumerate(b["classes"])}
    ext["label"] = ext["label_key"].map(lambda k: keymap.get(k.replace(" ", "_")))
    unmapped = sorted(ext[ext.label.isna()].label_key.unique()); ext = ext.dropna(subset=["label"]).assign(label=lambda d: d.label.astype(int))
    if unmapped: print(f"  {name}: classes not in the trained model were ignored: {unmapped}")
    if ext.label.nunique() < 2: print(f"  {name}: fewer than 2 matching classes - skipped"); continue
    y, p = predict_ds(b["model"], make_ds(ext, b["size"], BATCH_SIZE, False))
    m = compute_metrics(y, p, b["classes"], MODULES[name], "external"); save_json(m, WORK / "reports" / name / f"cross_dataset_{Path(path).name}.json")
    print(f"{name} on {Path(path).name}: accuracy={m['accuracy']:.3f} macro-F1={m['f1_macro']:.3f} (n={m['n']}) - compare with in-dataset test metrics")

In [ ]:
# ===== CELL 19: architecture comparison (uses VALIDATION data only; disabled by default) =====
COMPARE_ARCHS = []            # e.g. ["densenet121", "efficientnet_b0", "resnet50v2"]
COMPARE_MODULES = ["pneumonia"]
COMPARE_EPOCHS = (2, 3)
if not COMPARE_ARCHS: print("Architecture comparison: disabled (set COMPARE_ARCHS to run it; it retrains short models, so it can take a while).")
rows = []
for name in COMPARE_MODULES if COMPARE_ARCHS else []:
    if name not in PREP: print(f"{name}: no prepared data in this session - skipped"); continue
    prep = PREP[name]; spec = MODULES[name]; size = size_for(name); df = prep["df"]
    tr, va = df[df.split == "train"], df[df.split == "val"]
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.arange(len(prep["classes"])), y=tr["label"].values)))
    for arch in COMPARE_ARCHS:
        try:
            t0 = time.time(); keras.backend.clear_session()
            model, bb, used = build_model(arch, len(prep["classes"]), size, spec["flip"], USE_PRETRAINED)
            tds, vds = make_ds(tr, size, BATCH_SIZE, True), make_ds(va, size, BATCH_SIZE, False)
            bb.trainable = False; compile_model(model, 1e-3); fit_stage(model, tds, vds, cw, COMPARE_EPOCHS[0], WORK / "checkpoints" / "cmp_h.weights.h5", 2, 1)
            unfreeze_top(bb, UNFREEZE_FRACTION); compile_model(model, 1e-5); fit_stage(model, tds, vds, cw, COMPARE_EPOCHS[1], WORK / "checkpoints" / "cmp_f.weights.h5", 2, 1)
            yv, pv = predict_ds(model, vds); m = compute_metrics(yv, pv, prep["classes"], spec, "validation")
            rows.append(dict(module=name, arch=arch, pretrained=used, params_M=round(model.count_params() / 1e6, 1), val_accuracy=m["accuracy"], val_f1_macro=m["f1_macro"], minutes=round((time.time() - t0) / 60, 1)))
        except Exception as e: rows.append(dict(module=name, arch=arch, error=f"{type(e).__name__}: {e}"))
if rows:
    cmp_df = pd.DataFrame(rows); print(cmp_df.to_string(index=False)); cmp_df.to_csv(WORK / "reports" / "architecture_comparison.csv", index=False)
    print("Selection uses validation metrics only; the test split stays untouched.")

In [ ]:
# ===== CELL 20: model cards, final manifest and verified archive =====
MODULE_LIMITS = {
 "pneumonia": ["Trained on paediatric chest radiographs from a single source; behaviour on adults or other scanners is unknown.", "Bacterial and viral pneumonia share one label.", "Labels may contain errors; text markers or devices on the image can act as shortcuts."],
 "brain_tumor": ["Works on single 2D slices; clinical reading uses volumes and several sequences.", "The dataset is a compilation of several sources, so acquisition differences may leak into labels."],
 "diabetic_retinopathy": ["Grades are ordinal but the loss treats them as nominal; check quadratic weighted kappa and per-class recall.", "Image quality, illumination and field of view vary widely."],
 "breast_cancer": ["Trained on one small public breast ultrasound dataset (about 780 images from 600 women, collected in 2018): performance on other scanners, hospitals or populations is unknown.", "Image-level split: several images can come from the same patient, so metrics may be optimistic.", "Ultrasound images can contain text, calipers or measurement marks that a model may use as shortcuts (the dataset's mask files are ignored).", "Classifies whole images only (benign / malignant / normal); it is not a screening tool and does not replace mammography or clinical examination."]}
def fmt(v): return "n/a" if v is None else f"{v:.3f}"
def write_model_card(name):
    d = WORK / "models" / name; m = json.loads((d / "metrics.json").read_text()); cfg = json.loads((d / "model_config.json").read_text())
    t, v = m["test"], m["validation"]; lines = [f"# Model card - {MODULES[name]['title']} ({MODULES[name]['input']})", "",
        "**Intended use:** education and research demonstrations only. **Not** a medical device; not for diagnosis or treatment decisions.", "",
        "## Model", f"- Architecture: {cfg['arch']} ({'ImageNet-pretrained' if cfg['pretrained_weights_used'] else 'RANDOM initialisation - no pretrained weights were used'})",
        f"- Input: {cfg['input_size']}x{cfg['input_size']} RGB; classes: {', '.join(json.loads((d / 'class_mapping.json').read_text())['classes'])}",
        f"- Training: two-stage transfer learning, Adam, class-weighted cross-entropy, seed {cfg['seed']}, epochs run {cfg['epochs_run']}", "",
        "## Data", f"- Dataset: {cfg['dataset']}", f"- Split sizes: {cfg['counts']}", f"- Split notes: {'; '.join(cfg['split_notes'])}"]
    if cfg.get("subsampled"): lines.append("- WARNING: trained on a subsample (CPU fallback); do not treat these numbers as final.")
    lines += ["", "## Performance (held-out TEST split; validation shown separately)", "| metric | validation | test |", "|---|---|---|"]
    for k in ("accuracy", "precision_macro", "recall_macro", "f1_macro", "roc_auc", "quadratic_weighted_kappa", "sensitivity", "specificity"):
        if k in t or k in v: lines.append(f"| {k} | {fmt(v.get(k))} | {fmt(t.get(k))} |")
    if (d / "calibration.json").exists():
        c = json.loads((d / "calibration.json").read_text()); lines += ["", "## Calibration / uncertainty", f"- Temperature {c['temperature']:.2f} (fitted on validation); test ECE {c['ece_test_before']:.3f} -> {c['ece_test_after']:.3f}", f"- Prediction flagged uncertain on {c['uncertain_fraction_test']:.0%} of test images (heuristic thresholds)"]
    if (d / "robustness.json").exists():
        r = pd.DataFrame(json.loads((d / "robustness.json").read_text())["results"]); w = r[r.severity == 3].set_index("corruption")["accuracy_drop"].round(3).to_dict()
        lines += ["", "## Robustness", f"- Accuracy drop at the strongest corruption level: {w}"]
    lines += ["", "## Limitations"] + [f"- {x}" for x in MODULE_LIMITS[name]] + [
        "- Public-dataset results do not predict performance on other hospitals, devices or populations; no external validation unless stated above.",
        "- Confidence is not a probability of disease. Grad-CAM shows model attention, not proof of pathology.", "- Not evaluated for bias across age, sex or ethnicity."]
    (d / "MODEL_CARD.md").write_text("\n".join(lines)); return d / "MODEL_CARD.md"

for name in trained_modules(): print("model card:", write_model_card(name))
if trained_modules():
    write_manifest(RESULTS); path, n_files, mb = build_zip()
    print(f"\nArchive rebuilt and verified: {path} ({n_files} files, {mb:.1f} MB)")
    try:
        from IPython.display import FileLink, display; display(FileLink(str(path)))
    except Exception: pass
else: print("No trained modules -> no model cards and no archive.")

In [ ]:
# ===== CELL 21: example inference with the exported artifacts (also proves inference.py works) =====
if trained_modules():
    sys.path.insert(0, str(WORK)); import importlib, inference; importlib.reload(inference)
    n = trained_modules()[0]; bundle = inference.load_module(WORK / "models" / n)
    row = load_trained(n)["splits"].query("split == 'test'").iloc[0]
    print(f"module={n} | true label={row['label_name']}"); print(json.dumps(inference.predict(bundle, row["path"]), indent=2))
    print("quality check:", quality_check(n, row["path"]))
else: print("Nothing trained yet - no inference example.")

---
## Full-stack website (run after training)
The cells below package a **FastAPI back end + React front end** that serves the models you just exported, start it, give you a public link, and test it end to end.
Upload any JPG/PNG: choose a module or use **Auto-detect**. You get the prediction, calibrated confidence, class probabilities, Grad-CAM heatmap, uncertainty / image-quality warnings and a disclaimer.
The website uses the artifacts in `models/`, so it only offers modules that trained successfully. Educational / research use only. Please upload only public or anonymised images: the public link passes through a third-party relay.

In [ ]:
# ===== CELL 22: write the website files (back end + built front end) =====
import base64 as _b64, io as _io
WEB = WORK / "webapp"; (WEB / "static").mkdir(parents=True, exist_ok=True)
APP_PY = r'''# MediVision AI - web back end. Reads the artifacts exported by the training notebook (models/<module>/...).
# Educational/research use only. Not a medical diagnosis. Uploaded images are processed in memory and never stored.
import base64, io, json, os
from functools import lru_cache
from pathlib import Path

import numpy as np
import tensorflow as tf
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.responses import FileResponse
from fastapi.staticfiles import StaticFiles
from PIL import Image
from tensorflow import keras
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path(__file__).resolve().parent
MODELS = Path(os.environ.get("MODEL_DIR", ROOT.parent / "models"))
STATIC = ROOT / "static"
MAX_BYTES = 10 * 1024 * 1024
TITLES = {"pneumonia": ("Pneumonia", "Chest X-ray"), "brain_tumor": ("Brain Tumor", "Brain MRI"),
          "diabetic_retinopathy": ("Diabetic Retinopathy", "Retinal fundus"), "breast_cancer": ("Breast Cancer", "Breast ultrasound")}
UNAVAILABLE = "Model is currently unavailable. Please configure the model before analysis."
UNCERTAIN_CONF, UNCERTAIN_ENTROPY = 0.70, 0.60     # heuristic warning thresholds (not clinically validated)

app = FastAPI(title="MediVision AI", version="2.0")

def _json(path):
    return json.loads(path.read_text()) if path.exists() else None

def ready(name):
    d = MODELS / name
    return all((d / f).exists() for f in ("model.keras", "class_mapping.json", "preprocessing.json"))

@lru_cache(maxsize=4)
def bundle(name):
    d = MODELS / name
    model = keras.models.load_model(d / "model.keras", compile=False)
    gate = None
    if (d / "modality_gate.npz").exists():
        z = np.load(d / "modality_gate.npz"); gate = dict(mu=z["mean"], P=z["precision"], thr=float(z["threshold"]))
    return dict(model=model, gap=keras.Model(model.input, model.get_layer("gap").output), gate=gate,
                classes=_json(d / "class_mapping.json")["classes"], size=_json(d / "preprocessing.json")["input_size"],
                cal=_json(d / "calibration.json"), quality=_json(d / "quality_profile.json"), cfg=_json(d / "model_config.json") or {})

def read_image(data, filename):
    if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
        raise HTTPException(415, "Only JPG, JPEG and PNG files are supported.")
    if len(data) > MAX_BYTES:
        raise HTTPException(413, "File exceeds the 10 MB limit.")
    try:
        Image.open(io.BytesIO(data)).verify()
        img = Image.open(io.BytesIO(data))
        if img.format not in ("JPEG", "PNG"): raise ValueError("format")
        return img.convert("RGB")
    except Exception:
        raise HTTPException(400, "File is not a valid JPEG/PNG image.")

def to_input(img, size):      # same resize as training: tf.image.resize bilinear on raw 0-255 RGB
    return tf.image.resize(np.asarray(img, dtype=np.float32), (size, size)).numpy()[None]

def gate_score(b, x):         # Mahalanobis distance of the image's features / threshold (>1 = unlike this module's training images)
    g = b["gate"]
    if g is None: return None
    d = b["gap"](x, training=False).numpy()[0] - g["mu"]
    return float((d @ g["P"]) @ d) / g["thr"]

def quality(img, b):
    prof = b["quality"]
    if not prof: return dict(ok=True, issues=[])
    g = np.asarray(img.convert("L").resize((b["size"], b["size"])), dtype=np.float32)
    lap = -4 * g[1:-1, 1:-1] + g[:-2, 1:-1] + g[2:, 1:-1] + g[1:-1, :-2] + g[1:-1, 2:]
    mean, std, sharp = float(g.mean()), float(g.std()), float(lap.var()); w, h = img.size; issues, hard = [], False
    if min(w, h) < prof["min_side"]: issues.append(f"image is very small ({w}x{h})"); hard = True
    if std < prof["blank_std"]: issues.append("image looks blank"); hard = True
    if mean < prof["too_dark_mean"]: issues.append("image is unusually dark")
    if mean > prof["too_bright_mean"]: issues.append("image is unusually bright")
    if std < prof["low_contrast_std"]: issues.append("image has unusually low contrast")
    if sharp < prof["blurry_sharpness"]: issues.append("image looks blurry")
    return dict(ok=not hard, issues=issues)

def calibrate(p, T):
    z = np.log(np.clip(p, 1e-12, 1)) / T; z = z - z.max(); e = np.exp(z); return e / e.sum()

def entropy(p): return float(-(p * np.log(np.clip(p, 1e-12, 1))).sum())

def mc_mean(model, x, passes=20):     # Monte-Carlo dropout on the head only
    bb = model.get_layer("backbone"); i = model.layers.index(bb)
    pre = keras.Model(model.input, model.layers[i - 1].output)
    f = bb(pre(x, training=False), training=False); outs = []
    for _ in range(passes):
        y = f
        for l in model.layers[i + 1:]: y = l(y, training=True)
        outs.append(y.numpy()[0])
    return np.mean(outs, 0)

def gradcam(model, x):
    bb = model.get_layer("backbone"); i = model.layers.index(bb)
    pre = keras.Model(model.input, model.layers[i - 1].output); head = model.layers[i + 1:]
    with tf.GradientTape() as t:
        f = bb(pre(x, training=False), training=False); t.watch(f)
        y = f
        for l in head: y = l(y, training=False)
        idx = int(tf.argmax(y[0])); score = y[0, idx]
    g = t.gradient(score, f)
    cam = tf.nn.relu(tf.reduce_sum(f * tf.reduce_mean(g, (1, 2), keepdims=True), -1))[0].numpy()
    return y[0].numpy(), idx, (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

def png_url(arr):
    buf = io.BytesIO(); Image.fromarray(arr).save(buf, "PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

def display_pair(img, cam):
    disp = img.copy(); disp.thumbnail((512, 512))
    rgb = np.asarray(disp, dtype=np.float32) / 255
    c = np.asarray(Image.fromarray((cam * 255).astype(np.uint8)).resize(disp.size, Image.BILINEAR)) / 255.0
    heat = plt.cm.jet(c)[..., :3]
    return png_url((rgb * 255).astype(np.uint8)), png_url((np.clip(0.6 * rgb + 0.4 * heat, 0, 1) * 255).astype(np.uint8))

def analyse(disease, img):
    avail = [n for n in TITLES if ready(n)]
    if not avail: raise HTTPException(503, UNAVAILABLE)
    warnings, routing = [], None
    if disease == "auto":
        scores = {}
        for n in avail:
            b = bundle(n); s = gate_score(b, to_input(img, b["size"]))
            if s is not None: scores[n] = s
        if not scores: raise HTTPException(503, "Auto-detect needs the modality gates produced by the notebook's advanced cells. Please choose a module manually.")
        disease = min(scores, key=scores.get); ordered = sorted(scores.values())
        if scores[disease] > 1.0:
            raise HTTPException(422, "This image does not look like any supported scan type (chest X-ray, brain MRI, retinal fundus, breast ultrasound). No prediction was made.")
        routing = dict(chosen=disease, scores={n: round(v, 2) for n, v in scores.items()})
        if len(ordered) > 1 and ordered[1] < 1.2 * ordered[0]:
            warnings.append("Auto-detection is ambiguous between several modules; please confirm the module manually.")
    if disease not in TITLES: raise HTTPException(404, f"Unknown module '{disease}'.")
    if not ready(disease): raise HTTPException(503, UNAVAILABLE)
    b = bundle(disease); x = to_input(img, b["size"]); q = quality(img, b)
    if not q["ok"]: raise HTTPException(422, "Image quality is too low for analysis: " + "; ".join(q["issues"]) + ".")
    warnings += [f"Image quality: {i}." for i in q["issues"]]
    if routing is None:
        s = gate_score(b, x)
        if s is not None and s > 1.0:
            warnings.append(f"This image does not look like a typical {TITLES[disease][1]} image; the prediction may be unreliable.")
    raw, idx, cam = gradcam(b["model"], x)
    p = calibrate(raw, (b["cal"] or {}).get("temperature", 1.0)); n = len(p)
    if p.max() < UNCERTAIN_CONF: warnings.append("Low confidence: the model is not sure about this image.")
    if entropy(mc_mean(b["model"], x)) / np.log(n) > UNCERTAIN_ENTROPY: warnings.append("High uncertainty: repeated stochastic passes disagree.")
    uncertain = any(w.startswith(("Low confidence", "High uncertainty")) for w in warnings)
    orig_url, cam_url = display_pair(img, cam); title, inp = TITLES[disease]
    return dict(disease=disease, disease_title=title, prediction=b["classes"][idx], confidence=float(p[idx]),
                probabilities={c: float(v) for c, v in zip(b["classes"], p)}, original_image_url=orig_url, gradcam_image_url=cam_url,
                model=dict(architecture=b["cfg"].get("arch", "unknown"), task="Binary Classification" if n == 2 else "Multi-class Classification", input=inp, version="v1.0"),
                warnings=warnings, uncertain=uncertain, routing=routing, quality=q, calibrated=bool(b["cal"]))

@app.get("/api/health")
def health():
    return {"status": "ok", "modules": {n: ("available" if ready(n) else "unavailable") for n in TITLES}}

@app.get("/api/models")
def models():
    out = []
    for n, (title, inp) in TITLES.items():
        d = MODELS / n; cfg = _json(d / "model_config.json") or {}; met = (_json(d / "metrics.json") or {}).get("test") or {}
        cls = _json(d / "class_mapping.json")
        metrics = ({"accuracy": met.get("accuracy"), "precision": met.get("precision_macro"), "recall": met.get("recall_macro"),
                    "f1": met.get("f1_macro"), "roc_auc": met.get("roc_auc")} if met else None)
        out.append(dict(id=n, disease=title, input=inp, architecture=cfg.get("arch", "n/a"), num_classes=len(cls["classes"]) if cls else None,
                        dataset=cfg.get("dataset"), version="v1.0", status="available" if ready(n) else "unavailable", metrics=metrics))
    return out

@app.post("/api/predict")
async def predict(disease: str = Form("auto"), file: UploadFile = File(...)):
    return analyse(disease, read_image(await file.read(), file.filename or ""))

@app.post("/api/gradcam")
async def gradcam_only(disease: str = Form("auto"), file: UploadFile = File(...)):
    r = analyse(disease, read_image(await file.read(), file.filename or ""))
    return {k: r[k] for k in ("disease", "prediction", "gradcam_image_url", "warnings")}

if (STATIC / "index.html").exists():
    app.mount("/assets", StaticFiles(directory=STATIC / "assets"))

    @app.get("/{path:path}", include_in_schema=False)
    def spa(path: str):
        if path.startswith("api/"): raise HTTPException(404)
        f = STATIC / path
        return FileResponse(f if path and f.is_file() else STATIC / "index.html")
'''
STATIC_B64 = "..."
WEB_README = '''# MediVision AI website (FastAPI back end + built React front end)
Run locally (needs the `models/` folder next to this `webapp/` folder, as in the downloaded archive):

    pip install -r requirements.txt
    uvicorn app:app --port 8000        # then open http://localhost:8000

The back end reads models/<module>/ (model.keras, class_mapping.json, preprocessing.json, calibration.json, quality_profile.json, modality_gate.npz).
Set MODEL_DIR to use another folder. Images are processed in memory and never stored.
Educational/research use only. Not a medical diagnosis.
'''
(WEB / "app.py").write_text(APP_PY)
with zipfile.ZipFile(_io.BytesIO(_b64.b64decode(STATIC_B64))) as z: z.extractall(WEB / "static")
(WEB / "requirements.txt").write_text("fastapi\nuvicorn[standard]\npython-multipart\ntensorflow\npillow\nmatplotlib\nnumpy\n")
(WEB / "README.md").write_text(WEB_README)
print("website files written to", WEB, "->", sorted(p.name for p in WEB.iterdir()))

In [ ]:
# ===== CELL 23: start the website and (optionally) create a public link =====
import urllib.request, threading
START_TUNNEL = os.environ.get("MEDIVISION_NO_TUNNEL") is None     # needs Internet ON in Kaggle settings
PORT = 8000
SERVER = None

def start_website(port=PORT, tunnel=START_TUNNEL):
    global SERVER
    if not trained_modules():
        print("No trained modules -> website NOT started (every module would show as unavailable). Train at least one module first."); return None
    _need("fastapi"); _need("uvicorn"); _need("multipart", "python-multipart")
    if SERVER is not None and SERVER.poll() is None: SERVER.terminate(); time.sleep(1)
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": "", "MODEL_DIR": str(WORK / "models"), "TF_CPP_MIN_LOG_LEVEL": "2"}   # CPU inference: never fights training for the GPU
    log = open(WEB / "server.log", "w")
    SERVER = subprocess.Popen([sys.executable, "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", str(port)], cwd=str(WEB), env=env, stdout=log, stderr=subprocess.STDOUT)
    health = None
    for _ in range(120):
        time.sleep(1)
        if SERVER.poll() is not None: break
        try: health = json.loads(urllib.request.urlopen(f"http://127.0.0.1:{port}/api/health", timeout=3).read()); break
        except Exception: pass
    if health is None:
        print("Server did not start. Last log lines:\n", (WEB / "server.log").read_text()[-2500:]); return None
    print("Website back end is running locally:", health)
    if tunnel:
        try:
            cf = WORK.parent / "cloudflared"
            if not cf.exists():
                urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", str(cf)); os.chmod(cf, 0o755)
            tun = subprocess.Popen([str(cf), "tunnel", "--url", f"http://localhost:{port}"], stderr=subprocess.PIPE, text=True); holder = {}
            def pump():
                for line in tun.stderr:
                    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
                    if m and "url" not in holder: holder["url"] = m.group(0)
            threading.Thread(target=pump, daemon=True).start()
            for _ in range(45):
                if "url" in holder: break
                time.sleep(1)
            print("\n>>> OPEN YOUR WEBSITE:", holder.get("url", "(the tunnel did not report a URL - check Internet is ON)"))
            print("    Keep this notebook session running while you use it. First prediction per module is slower (model loading).")
        except Exception as e:
            print(f"Could not create a public link ({type(e).__name__}: {e}). Turn Internet ON, or run the site on your computer (see webapp/README.md in the archive).")
    return SERVER

start_website()

In [ ]:
# ===== CELL 24: end-to-end self-test of the running website =====
_need("requests"); import requests
BASE = f"http://127.0.0.1:{PORT}"
if SERVER is None or SERVER.poll() is not None: print("Website is not running (run cell 23 first).")
else:
    n = trained_modules()[0]; row = load_trained(n)["splits"].query("split == 'test'").iloc[0]; path = row["path"]
    def call(disease, content=None, fname=None):
        data = content if content is not None else open(path, "rb").read()
        return requests.post(BASE + "/api/predict", data={"disease": disease}, files={"file": (fname or Path(path).name, data)}, timeout=600)
    checks = []
    r = requests.get(BASE + "/api/models", timeout=30); checks.append(("GET /api/models", r.status_code == 200 and len(r.json()) == 4))
    r = call(n); ok = r.status_code == 200 and {"prediction", "confidence", "probabilities", "gradcam_image_url", "warnings"} <= set(r.json())
    checks.append((f"POST /api/predict ({n})", ok)); print("  ->", {k: r.json().get(k) for k in ("prediction", "confidence", "warnings")} if r.status_code == 200 else r.text[:300])
    r = call("auto"); checks.append(("POST /api/predict (auto-detect)", r.status_code in (200, 422, 503)))
    print("  auto-detect ->", (r.json().get("routing") or r.json().get("detail")) if r.headers.get("content-type", "").startswith("application/json") else r.text[:200])
    r = call(n, content=b"this is not an image", fname="x.jpg"); checks.append(("invalid file is rejected (400)", r.status_code == 400))
    r = call(n, content=b"x", fname="x.txt"); checks.append(("unsupported type is rejected (415)", r.status_code == 415))
    r = requests.get(BASE + "/", timeout=30); checks.append(("front end served at /", r.status_code == 200 and "<div id=\"root\">" in r.text))
    for name, ok in checks: print(("PASS  " if ok else "FAIL  ") + name)

In [ ]:
# ===== CELL 25: final package (models + reports + website) =====
write_manifest(RESULTS); path, n_files, mb = build_zip()
print(f"Archive rebuilt and verified: {path} ({n_files} files, {mb:.1f} MB) - includes models/, reports/, webapp/ and inference.py")
print("To run the website on your own computer: unzip it, `pip install -r webapp/requirements.txt`, `cd webapp`, `uvicorn app:app --port 8000`, open http://localhost:8000")
print("To stop the website here: SERVER.terminate()")
try:
    from IPython.display import FileLink, display; display(FileLink(str(path)))
except Exception: pass